In [ ]:
# requests: Used to send HTTP requests to web pages.
# BeautifulSoup: Parses the HTML content so we can extract specific elements like links.
from bs4 import BeautifulSoup
import pandas as pd
import logging
import re
from openai import OpenAI
import os
import json
import streamlit as st

In [ ]:

#grabe category and recipe name
category_and_recipe = dict()
recipe_ingredients_direction = dict()
recipe_item_dict = {}
allrecipes_existing_json_file = None 

In [ ]:
if os.path.exists(r'C:\Users\alam\OneDrive - New York State Thruway Authority\Documents\Python\allrecipes\allrecipe_data.json'):
    with open(r'C:\Users\alam\OneDrive - New York State Thruway Authority\Documents\Python\allrecipes\allrecipe_data.json') as file:
        existing_data = json.load(file)
        allrecipes_existing_json_file = existing_data

In [ ]:
for item in allrecipes_existing_json_file:
    print(item)

In [ ]:
#Extract ingredients 
import requests 
from bs4 import BeautifulSoup
def extract_ingredinets (link):
# url ="https://www.allrecipes.com/recipe/23891/grilled-cheese-sandwich/"
    response = requests.get(link)
    soup = BeautifulSoup(response.text, "html.parser")
    name = soup.find("h2", string ='Ingredients')
    ingredients_list = []
    if name:
        for li in name.find_all_next('li', class_='mm-recipes-structured-ingredients__list-item'):
            text = ' '.join(span.text for span in li.find_all('span'))
            ingredients_list.append(text)
    return ingredients_list


In [ ]:
import requests 
from bs4 import BeautifulSoup
def extract_direction(link):
    response = requests.get(link)
    soup = BeautifulSoup(response.text, "html.parser")
    name = soup.find("h2", string ='Directions')
    direction_list =[]
    if name:
        for li in name.find_all_next('p', class_='comp mntl-sc-block mntl-sc-block-html'):
            text = li.get_text(strip=True)
            direction_list.append(text)
    return direction_list

In [ ]:
#append the data into dictionary
def final_generate_file (daily_meal_name):
    if daily_meal_name not in allrecipes_existing_json_file:
       allrecipes_existing_json_file[daily_meal_name] =[]
       allrecipes_existing_json_file[daily_meal_name].append(recipe_ingredients_direction)
     
     


In [ ]:
#extract recipe name and ingredients
def extract_recipe_name_and_link (daily_meal_name,category,link):
    response = requests.get(link)
    soup = BeautifulSoup(response.text, "html.parser")
    titles = soup.find_all("span", class_="card__title-text")
    
    for title in titles:
        recipe_name = title.get_text(strip=True)
        parent_link = title.find_parent("a")
        if parent_link and parent_link.has_attr("href"):
            parent_link = parent_link['href']
            ingredients = extract_ingredinets(parent_link)
            direction= extract_direction(parent_link)
        #    # Append the recipe to the correct meal category
            if category not in recipe_ingredients_direction:
                recipe_ingredients_direction[category] = []
                
            recipe_ingredients_direction[category].append({
                "Recipe": recipe_name,
                "Ingredients": ingredients,
                "Direction":direction
                })
            
    final_generate_file (daily_meal_name)   

In [ ]:
def send_all_recipe():
    for category, items in recipe_item_dict.items():
        print(f"\nCategory: {category}")
        for item in items[:10]:  # Only take the first 2 items
            print(item['name'],item['link'])
            extract_recipe_name_and_link (category,item['name'],item['link'])

In [ ]:
#Grab All Recipe Link
import requests 
from bs4 import BeautifulSoup
def grab_all_recipe_item(recipe_name,base_url):
    response = requests.get(base_url)
    soup = BeautifulSoup(response.text, "html.parser")
    start_tag = soup.find("h1")
    ul_tag = start_tag.find_next('ul')
    if ul_tag:
        for li in ul_tag.find_all('li'):
                a_tag=li.find('a')
                if a_tag:
                     name = a_tag.get_text(strip=True)
                     link =a_tag.get('href')
                if recipe_name not in recipe_item_dict:
                    recipe_item_dict[recipe_name] = []
                if isinstance(link, str) and link.startswith("http"):
                    recipe_item_dict[recipe_name].append({
                        "name":name,
                        "link":link
                    })
    
    send_all_recipe()
   


In [ ]:
# Scrape from website all Recipe Data
import requests 
from bs4 import BeautifulSoup
url = "https://www.allrecipes.com/recipes/"
recipe = dict()
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")
start_tag = soup.find("h1")
end_tag = soup.find("h2")
current = start_tag
while current and current!=end_tag:
    current = current.find_next()
    if current == end_tag:
        break
    if current.name == "a":
        span = current.find("span")
        recipes_name = span.get_text(strip=True)
        link=current["href"]
        if isinstance(link, str) and link.startswith("http"):
            recipe[recipes_name] = link
del recipe["Ingredients"]
for recipe_name, link in recipe.items():
    print(recipe_name, link)
    grab_all_recipe_item(recipe_name,link)


       

In [ ]:
name = 'Salad Recipes'
url = "https://www.allrecipes.com/recipes/96/salad/"
grab_all_recipe_item(name,url)

In [ ]:
# save to json file into local pc
with open(r'C:\Users\alam\OneDrive - New York State Thruway Authority\Documents\Python\allrecipes\allrecipe_data.json', "w") as file:
    json.dump(allrecipes_existing_json_file, file, indent=4)
